In [47]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])
ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')

###############################################
                ## ESTIMATION ##
###############################################

#####################
      ## OLS ##
#####################

def OLS(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        revisions = df[f'r_t{j}']
        revisions = sm.add_constant(revisions)
        errors = df[f'e_t{j}']
        initial = sm.OLS(errors, revisions).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=1))
    return regs

ind_regs = OLS(ind_spf_trim, '2016-12-31')
ind_regs[3].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                  0.014
Method:                 Least Squares   F-statistic:                     20.71
Date:                Sat, 20 Apr 2024   Prob (F-statistic):           5.54e-06
Time:                        16:08:21   Log-Likelihood:                 10233.
No. Observations:                3424   AIC:                        -2.046e+04
Df Residuals:                    3422   BIC:                        -2.045e+04
Df Model:                           1                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0023      0.000     -8.002      0.000      -0.003      -0.002
r_t3          -0.2021      0.044     -4.550      0.000      -0.289      -0.115
==============================================================================
Omnibus:                      116.815   Durbin-Watson:                   0.406
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              219.697
Skew:                          -0.257   Prob(JB):                     1.97e-48
Kurtosis:                       4.129   Cond. No.                         140.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""

In [48]:
mean_regs = OLS(mean_spf_trim, '2016-12-31')
mean_regs[3].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     1.532
Date:                Sat, 20 Apr 2024   Prob (F-statistic):              0.218
Time:                        16:08:24   Log-Likelihood:                 440.50
No. Observations:                 141   AIC:                            -877.0
Df Residuals:                     139   BIC:                            -871.1
Df Model:                           1                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0022      0.001     -1.750      0.082      -0.005       0.000
r_t3           0.2573      0.208      1.238      0.218      -0.154       0.668
==============================================================================
Omnibus:                       10.826   Durbin-Watson:                   0.640
Prob(Omnibus):                  0.004   Jarque-Bera (JB):               15.671
Skew:                          -0.412   Prob(JB):                     0.000395
Kurtosis:                       4.410   Cond. No.                         219.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""

In [49]:
#####################
 ## Ind Forecasts ##
#####################

### Pooled OLS ###
ind_spf_trim = ind_spf_trim.loc[(ind_spf_trim['DATE'] >= '1981-09-30') & (ind_spf_trim['DATE'] <= a)]  # Filter data

ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')
ind_est_pld = est_table(ind_spf_trim)

### ID Fixed Effects ###
def est_table_fe(df):
    est_table = pd.DataFrame(index=['const', 'beta_1'])
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        reg = initial.get_robustcov_results(cov_type='HAC', maxlags=1, hasconst=True)
        est_table[f'coef_t{j}'] = reg.params[:2]
        est_table[f'std_err{j}'] = reg.bse[:2]
        est_table[f'tval{j}'] = reg.tvalues[:2]
        est_table[f'pval{j}'] = reg.pvalues[:2]
        est_table[f'nobs{j}'] = initial.nobs
    return est_table

ind_est_fe = est_table_fe(ind_spf_trim)

### Two-way Fixed Effects ###
def est_table_fe2(df):
    est_table = pd.DataFrame(index=['const', 'beta_1'])
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float), pd.get_dummies(ind_spf_trim['DATE'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        reg = initial.get_robustcov_results(cov_type='HAC', maxlags=1, hasconst=True)
        est_table[f'coef_t{j}'] = reg.params[:2]
        est_table[f'std_err{j}'] = reg.bse[:2]
        est_table[f'tval{j}'] = reg.tvalues[:2]
        est_table[f'pval{j}'] = reg.pvalues[:2]
        est_table[f'nobs{j}'] = initial.nobs
    return est_table

ind_est_fe2 = est_table_fe2(ind_spf_trim)

#####################
    ## AR(1) ##
#####################

vintage_trim = vintage_trim.loc[(vintage_trim['DATE'] >= '1965-06-30') & (vintage_trim['DATE'] <= a)]  # Filter data

def ar_j(df):
    ar_table = pd.DataFrame()
    grwth_t = df['t3']
    reg = AutoReg(grwth_t, 1).fit()
    ar_table.loc['const', 'coef'] = reg.params[0]
    ar_table.loc['phi', 'coef'] = reg.params[1]
    ar_table.loc['rho', 'coef'] = reg.roots
    return ar_table

ar_table = ar_j(vintage_trim)

###############################################
            ## MODEL PARAMETERS ##
###############################################

def params(dfm, dfp, dff, dff2, ar):
    params = pd.DataFrame(index=['ols', 'pld', 'fe', 'fe2'])
    params.loc['ols', 'lambda'] = dfm.loc['beta_1', 'coef_t3'] / (1 + dfm.loc['beta_1', 'coef_t3'])
    params.loc['ols', 'G'] = 1 / (1 + dfm.loc['beta_1', 'coef_t3'])
    pldc = dfp.iloc[1]['coef_t3']
    fec = dff.iloc[1]['coef_t3']
    fe2c = dff2.iloc[1]['coef_t3']
    ar1 = ar.loc['const', 'coef']
    params.loc['pld', 'Theta'] = (-((2 * pldc) + 1) + np.sqrt(((2 * pldc) + 1)**2 - 4 * (pldc + (pldc * ar1**2) + 1) * pldc)) / (2 * (pldc + (pldc * ar1**2) + 1))
    params.loc['fe', 'Theta'] = (-((2 * fec) + 1) + np.sqrt(((2 * fec) + 1)**2 - 4 * (fec + (fec * ar1**2) + 1) * fec)) / (2 * (fec + (fec * ar1**2) + 1))
    params.loc['fe2', 'Theta'] = (-((2 * fe2c) + 1) + np.sqrt(((2 * fe2c) + 1)**2 - 4 * (fe2c + (fe2c * ar1**2) + 1) * fe2c)) / (2 * (fe2c + (fe2c * ar1**2) + 1))
    return params

parameters = params(mean_estimations, ind_est_pld, ind_est_fe, ind_est_fe2, ar_table)

###############################################
                ## EXPORT ##
###############################################

mean_estimations.to_csv('output/mean_estimations.csv', sep=',', index=True)
ind_est_pld.to_csv('output/ind_est_pld.csv', sep=',', index=True)
ind_est_fe.to_csv('output/ind_est_fe.csv', sep=',', index=True)
ind_est_fe2.to_csv('output/ind_est_fe2.csv', sep=',', index=True)
ar_table.to_csv('output/ar_estimations.csv', sep=',', index=True)
parameters.to_csv('output/parameters.csv', sep=',', index=True)

NameError: name 'a' is not defined